# 🏠 Image-Based Classification of Indonesian Traditional Houses

**Deep Learning approach using FastAI & ResNet50 for classifying 5 types of Indonesian traditional houses.**

| Item | Detail |
|------|--------|
| **Task** | Multi-class Image Classification |
| **Framework** | FastAI v2 + PyTorch |
| **Backbone** | ResNet-50 (pretrained on ImageNet) |
| **Classes** | Balinese, Batak, Dayak, Javanese, Minangkabau |
| **Dataset** | Indonesian Traditional Houses Dataset |

---

## 📑 Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Configuration & Data Loading](#2-configuration--data-loading)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis-eda)
4. [Data Augmentation & DataLoaders](#4-data-augmentation--dataloaders)
5. [Model Architecture & Training](#5-model-architecture--training)
6. [Evaluation](#6-evaluation)
7. [Inference on Test Set](#7-inference-on-test-set)
8. [Export & Save Model](#8-export--save-model)

---
## 1. Environment Setup

Install and import all required libraries.

In [ ]:
# Install FastAI (uncomment if not installed)
# !pip install fastai

In [ ]:
# Core libraries
import os
import warnings
warnings.filterwarnings('ignore')

# FastAI
from fastai.vision.all import *

# Data manipulation & visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print(f'FastAI version : {fastai.__version__}')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')

---
## 2. Configuration & Data Loading

Define dataset paths and verify directory structure.

> **Note:** The dataset should be placed inside `Indonesian Traditional Houses Dataset/` relative to this notebook.

In [ ]:
# ============================================================
# Configuration
# ============================================================
TRAIN_PATH = Path('Indonesian Traditional Houses Dataset/Train/Train')
TEST_PATH  = Path('Indonesian Traditional Houses Dataset/Test/Test')
MODEL_DIR  = Path('models')

# Create model output directory
os.makedirs(MODEL_DIR, exist_ok=True)

# Verify paths
assert TRAIN_PATH.exists(), f'Training path not found: {TRAIN_PATH}'
assert TEST_PATH.exists(),  f'Test path not found: {TEST_PATH}'

# List available classes
classes = sorted([d.name for d in TRAIN_PATH.iterdir() if d.is_dir()])
print(f'Classes found: {classes}')
print(f'Number of classes: {len(classes)}')

---
## 3. Exploratory Data Analysis (EDA)

Understand the dataset distribution before training.

In [ ]:
# Count images per class
class_counts = {}
for cls in classes:
    cls_path = TRAIN_PATH / cls
    count = len(list(cls_path.glob('*')))
    class_counts[cls] = count

df_dist = pd.DataFrame({
    'Class': list(class_counts.keys()),
    'Count': list(class_counts.values())
}).sort_values('Count', ascending=True)

total_images = df_dist['Count'].sum()
print(f'Total training images: {total_images:,}')
print(f'Test images          : {len(list(TEST_PATH.glob("*"))):,}')
print('\nDistribution per class:')
print(df_dist.to_string(index=False))

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(figsize=(10, 5))

colors = sns.color_palette('Set2', len(df_dist))
bars = ax.barh(df_dist['Class'], df_dist['Count'], color=colors, edgecolor='white', linewidth=0.8)

# Add count labels
for bar, count in zip(bars, df_dist['Count']):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2,
            f'{count}', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Number of Images', fontsize=12)
ax.set_title('Training Dataset — Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlim(0, df_dist['Count'].max() * 1.15)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

# Note on class imbalance
max_cls = df_dist.iloc[-1]
min_cls = df_dist.iloc[0]
print(f'\n⚠️  Class imbalance detected: '
      f"{max_cls['Class']} ({max_cls['Count']}) vs {min_cls['Class']} ({min_cls['Count']}) "
      f"— ratio {max_cls['Count'] / min_cls['Count']:.1f}x")

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(1, len(classes), figsize=(20, 4))
for ax, cls in zip(axes, classes):
    cls_path = TRAIN_PATH / cls
    img_path = list(cls_path.glob('*'))[0]
    img = plt.imread(str(img_path))
    ax.imshow(img)
    ax.set_title(cls.capitalize(), fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Image per Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Data Augmentation & DataLoaders

Create FastAI `DataLoaders` with:
- **Resize** to 320×320
- **Standard augmentations** (flip, rotate, zoom, warp, lighting)
- **80/20** train/validation split with fixed seed for reproducibility

In [ ]:
# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
set_seed(SEED)

# Build DataLoaders
dls = ImageDataLoaders.from_folder(
    TRAIN_PATH,
    train='.',
    valid_pct=0.2,
    seed=SEED,
    item_tfms=Resize(320),
    batch_tfms=aug_transforms(),
    num_workers=4
)

print(f'Classes       : {dls.vocab}')
print(f'Num classes   : {dls.c}')
print(f'Train batches : {len(dls.train)}')
print(f'Valid batches : {len(dls.valid)}')

In [ ]:
# Preview augmented training batch
dls.show_batch(max_n=9, figsize=(8, 7))

---
## 5. Model Architecture & Training

Transfer learning with **ResNet-50** pretrained on ImageNet.

**Strategy:**
1. Train head layers (frozen backbone) for 20 epochs
2. Fine-tune entire network (unfrozen) for 10 epochs with discriminative learning rates

In [ ]:
# Create learner
learn = vision_learner(
    dls,
    resnet50,
    metrics=accuracy,
    model_dir=MODEL_DIR,
    path=Path('.')
)

print(f'Model: ResNet-50')
print(f'Device: {learn.dls.device}')

In [ ]:
# Find optimal learning rate
lr_suggestion = learn.lr_find()

In [ ]:
# Stage 1: Train head layers (backbone frozen)
lr1 = 1e-4
lr2 = 1e-3

print('Stage 1: Training head layers (backbone frozen)')
print(f'Learning rate range: {lr1} → {lr2}')
print(f'Epochs: 20')
print('=' * 60)

learn.fit_one_cycle(20, slice(lr1, lr2))

In [ ]:
# Stage 2: Fine-tune entire network (unfrozen)
learn.unfreeze()

print('Stage 2: Fine-tuning entire network (unfrozen)')
print(f'Learning rate range: 1e-5 → 1e-4')
print(f'Epochs: 10')
print('=' * 60)

learn.fit_one_cycle(10, slice(1e-5, 1e-4))

In [ ]:
# Plot training & validation loss
learn.recorder.plot_loss()

---
## 6. Evaluation

Analyze model performance on the validation set using:
- **Confusion Matrix**
- **Top Losses** — images the model is most confused about

In [ ]:
# Confusion matrix
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(8, 7))

In [ ]:
# Top losses — most confused predictions
interp.plot_top_losses(6, figsize=(25, 5))

---
## 7. Inference on Test Set

Generate predictions for the unlabeled test images.

In [ ]:
# Create test DataLoader
test_dl = learn.dls.test_dl(get_image_files(TEST_PATH))
print(f'Test images: {len(test_dl.dataset)}')

In [ ]:
# Run inference
preds, _ = learn.get_preds(dl=test_dl)

pred_classes = preds.argmax(dim=1)
pred_probs   = preds.max(dim=1).values

print(f'Predictions generated: {len(pred_classes)}')
print(f'Average confidence   : {pred_probs.mean():.4f}')

In [ ]:
# Build results DataFrame
test_files = get_image_files(TEST_PATH)

df_results = pd.DataFrame({
    'id': [f.stem for f in test_files],
    'style': [learn.dls.vocab[i] for i in pred_classes],
    'confidence': [f'{p:.4f}' for p in pred_probs]
})

df_results = df_results.sort_values('id').reset_index(drop=True)

print(f'Results shape: {df_results.shape}')
print(f'\nPrediction distribution:')
print(df_results['style'].value_counts().to_string())
print(f'\nFirst 10 predictions:')
df_results.head(10)

In [ ]:
# Save predictions to CSV
output_path = 'predictions.csv'
df_results[['id', 'style']].to_csv(output_path, index=False)
print(f'✅ Predictions saved to: {output_path}')

---
## 8. Export & Save Model

Save the trained model for future inference.

In [ ]:
# Export complete model (architecture + weights + vocab)
export_path = MODEL_DIR / 'export.pkl'
learn.export(fname=export_path)
print(f'✅ Model exported to: {export_path}')

# Save checkpoint
learn.model_dir = str(MODEL_DIR)
saved_path = learn.save('stage-final')
print(f'✅ Checkpoint saved to: {MODEL_DIR}/stage-final.pth')

---

## 📝 Summary

| Stage | Description | Epochs |
|-------|-------------|--------|
| 1 | Train head (frozen backbone) | 20 |
| 2 | Fine-tune full network | 10 |

**Model:** ResNet-50 (pretrained ImageNet) → Transfer Learning → Indonesian Traditional Houses  
**Output:** `predictions.csv` + `models/export.pkl`

---
*Notebook by Aditama — [GitHub Repository](https://github.com/)*